code by Huan Li

In [ ]:
# paths.py
from pathlib import Path

PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
SINGLE_COLUMN_DIR = PROJECT_ROOT / "data"/ "speeches" / "txt_from_image"/ "single_column"
PDF_DIR = PROJECT_ROOT / "data" / "dataframes" / "metadata_s05.xlsx"

In [ ]:
import pandas as pd
metadata_s05 =  pd.read_excel(PDF_DIR)
metadata_s05.head()

In [ ]:
txt_files = sorted(SINGLE_COLUMN_DIR.glob("single_batch_*.txt"))

for txt_path in txt_files:
    with txt_path.open("r", encoding="utf-8") as f:
        text = f.read()

    print(txt_path.name, len(text))

In [ ]:
import re

# 1) pdf_names
pdf_names = metadata_s05["pdf_name"].tolist()

# 2) input txt dir
batch_dir =  PROJECT_ROOT / "data"/ "speeches" / "txt_from_image"/ "single_column"
batch_paths = [batch_dir / f"single_batch_{i:02d}.txt" for i in range(1, 10)]

# 3) output dir
out_dir = PROJECT_ROOT / "data" / "speeches" / "extracted_from_image"
out_dir.mkdir(parents=True, exist_ok=True)

# 4) regex patterns
start_sg = re.compile(
    r"(?im)^\s*(?:\(?\s*)?(?:\d+\s*[\.\)]\s*)?(?:the\s+)?"
    r"(?:secreta(?:r|i|l|v)y(?:\s*[\-\u2010\u2011\u2012\u2013\u2014]\s*|\s+)general)"
    r"(?:\s*\([^)]*\))?\s*[:;]\s*"
)

next_speaker = re.compile(
    r"(?im)^\s*(?:\(?\s*)?(?:\d+\s*[\.\)]\s*)?"
    r"(?:The\s+(?:President|Chairman|Chair|Rapporteur|Secretary(?:\s*[\-\u2010-\u2014]\s*General|(?:\s+General)?)|Representative)"
    r"|(?:Mr|Mrs|Ms|Miss|Mme|M|Sir|Dr|Ambassador|Minister|Prince|Princess)\.?)"
    r"(?:\s+[A-Z][\w'\-\.]{1,30}){0,6}(?:\s*\([^)]*\)){0,4}\s*[:;]\s*"
)

# 5) read all txt files at once (concatenate in order 01->09)
big_text = "\n\n".join(p.read_text(encoding="utf-8", errors="replace") for p in batch_paths)

# 6) extract all SG speeches at once (in order of appearance)
speeches = []
for m in start_sg.finditer(big_text):
    start = m.end()
    stop = next_speaker.search(big_text, start)
    end = stop.start() if stop else len(big_text)
    speeches.append((m.group().strip(), big_text[start:end].strip()))

print("Found SG speeches:", len(speeches))

# 7) alignment check + save with sequential naming
if len(speeches) != len(pdf_names):
    raise ValueError(f"Mismatch: found {len(speeches)} speeches, but have {len(pdf_names)} pdf_names")

name_count = {}
for i, (header, body) in enumerate(speeches):
    stem = Path(pdf_names[i]).stem
    name_count[stem] = name_count.get(stem, 0) + 1
    suffix = f"__{name_count[stem]:02d}" if name_count[stem] > 1 else ""
    (out_dir / f"{stem}{suffix}.txt").write_text(header + "\n" + body, encoding="utf-8")

print("Done. Saved to:", out_dir)


In [ ]:
print(f"Found SG speeches: {len(speeches)}")
print(f"Have pdf_names: {len(pdf_names)}\n")


for i, (header, body) in enumerate(speeches):
    name = pdf_names[i] if i < len(pdf_names) else "NO_NAME"
    preview = body.replace("\n", " ")[:200]
    print(f"[{i:02d}] {name}")
    print(f"  header : {header}")
    print(f"  len    : {len(body)}")
    print(f"  preview: {preview}\n")
